# PyTorch Training Loop Practice

**Topic:** PyTorch Basics &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_train` `(2000, 2)`, `y_train` `(2000,)`: noisy samples of a smooth 2-D surface `y = sin(2x₀) · cos(x₁) + 0.5·x₁` (NumPy arrays, so convert them to tensors with `torch.tensor(..., dtype=torch.float32)`).
- `X_test` `(600, 2)`: predict these. `y_test` is used by the evaluation cell.

## Your task

1. Build a small MLP, e.g. `nn.Sequential(nn.Linear(2, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))`.
2. Train with `torch.optim.Adam` on MSE loss for a few thousand steps.
3. Convert your test outputs back to NumPy and set `predictions` (shape `(600,)`).

## Evaluation

Test R² must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

def make(n, seed):
    r = np.random.default_rng(seed)
    X = r.uniform(-2, 2, (n, 2))
    y = np.sin(2 * X[:, 0]) * np.cos(X[:, 1]) + 0.5 * X[:, 1] + r.normal(0, 0.1, n)
    return X, y

X_train, y_train = make(2000, 211)
X_test, y_test = make(600, 212)
print(f"X_train {X_train.shape}. Convert to torch tensors in your solution")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_R2 = 0.9

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions, dtype=float).ravel()
_true = np.asarray(y_test, dtype=float).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

_rmse = float(np.sqrt(np.mean((_pred - _true) ** 2)))
_ss_res = float(np.sum((_true - _pred) ** 2))
_ss_tot = float(np.sum((_true - _true.mean()) ** 2))
r2 = 1.0 - _ss_res / _ss_tot
print("=" * 52)
print(f"Test RMSE : {_rmse:.4f}")
print(f"Test R^2  : {r2:.4f}   (target: >= {TARGET_R2:.2f})")
print(f"Worst absolute error: {float(np.max(np.abs(_pred - _true))):.4f}")
print("=" * 52)
if r2 >= TARGET_R2:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

This is the intended solution: torch tensors, the `nn.Sequential` MLP from the
task, Adam on MSE, and the training loop.

```python
import torch
import torch.nn as nn

Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
Xte = torch.tensor(X_test, dtype=torch.float32)

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 64), nn.Tanh(),
                      nn.Linear(64, 64), nn.Tanh(),
                      nn.Linear(64, 1))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for _ in range(3000):
    loss = loss_fn(model(Xtr), ytr)
    opt.zero_grad()
    loss.backward()
    opt.step()

model.eval()
predictions = model(Xte).detach().numpy().ravel()
```

<details><summary>No torch installed? A scikit-learn fallback</summary>

`MLPRegressor(hidden_layer_sizes=(64, 64), activation="tanh", max_iter=5000)` is
a reasonable stand-in for reading the code, but it is not the assigned exercise
and was not checked to reach the R^2 target.

```python
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
mlp = MLPRegressor(hidden_layer_sizes=(64, 64), activation="tanh",
                   max_iter=5000, random_state=0)
mlp.fit(scaler.transform(X_train), y_train)
predictions = mlp.predict(scaler.transform(X_test))
```

</details>
